# Hamshahri Word and 3-gram Index - Without Stop Words

Build a SQLite inverted index from the text documents.It ignores stopwords and then normalize the text by Hazm library. The `inverted_index` table stores  three-word sequences as primary keys in `ngram`. Its `Total` column contains each matching total occurrence count (file1(n)+file2(n)+...). Show top 20 of rows sorted by 'Total' Desc

In [6]:
import json
import re
import sqlite3
from pathlib import Path

from hazm import Normalizer, word_tokenize

CORPUS_DIR = Path("data/Hamshahri_data")
STOPWORDS_PATH = Path("data/stopwords-fa.txt")
DATABASE_PATH = Path("data/hamshahri_3gram.db")
WORD_PATTERN = re.compile(r"[^\W_]+", re.UNICODE)
NORMALIZER = Normalizer()


def extract_strings(value):
    if isinstance(value, str):
        yield value
    elif isinstance(value, dict):
        for child in value.values():
            yield from extract_strings(child)
    elif isinstance(value, list):
        for child in value:
            yield from extract_strings(child)


def read_document_text(path):
    content = path.read_text(encoding="utf-8")
    try:
        document = json.loads(content)
    except json.JSONDecodeError:
        return content
    return " ".join(extract_strings(document))


def tokenize_words(text):
    normalized = NORMALIZER.normalize(text)
    return [
        token.casefold()
        for token in word_tokenize(normalized)
        if WORD_PATTERN.fullmatch(token)
    ]


with STOPWORDS_PATH.open(encoding="utf-8") as stopword_file:
    STOPWORDS = {
        token
        for line in stopword_file
        for token in tokenize_words(line.strip())
    }

In [7]:
def build_inverted_index(corpus_dir=CORPUS_DIR, database_path=DATABASE_PATH):
    """Build a stopword-filtered trigram index with per-file and total counts."""
    from collections import Counter

    corpus_dir = Path(corpus_dir)
    database_path = Path(database_path)
    documents = sorted(corpus_dir.rglob("*.txt"))
    if not documents:
        raise FileNotFoundError(f"No .txt files found in {corpus_dir.resolve()}")

    index = {}
    for path in documents:
        tokens = [
            token for token in tokenize_words(read_document_text(path))
            if token not in STOPWORDS
        ]
        filename = path.relative_to(corpus_dir).as_posix()
        trigram_counts = Counter(
            " ".join(tokens[position : position + 3])
            for position in range(len(tokens) - 2)
        )
        for trigram, count in trigram_counts.items():
            index.setdefault(trigram, []).append((filename, count))

    database_path.parent.mkdir(parents=True, exist_ok=True)
    with sqlite3.connect(database_path) as connection:
        connection.execute("DROP TABLE IF EXISTS inverted_index")
        connection.execute(
            "CREATE TABLE inverted_index ("
            "ngram TEXT PRIMARY KEY NOT NULL, "
            "filenames TEXT NOT NULL, "
            "Total INTEGER NOT NULL)"
        )

        rows = []
        for trigram, file_counts in index.items():
            ordered_files = sorted(
                file_counts, key=lambda item: (-item[1], item[0])
            )
            filenames = "+".join(
                f"{filename}({count})"
                for filename, count in ordered_files
            )
            total = sum(count for _, count in file_counts)
            rows.append((trigram, filenames, total))

        connection.executemany(
            "INSERT INTO inverted_index (ngram, filenames, Total) "
            "VALUES (?, ?, ?)",
            rows,
        )

    print(
        f"Indexed {len(documents)} documents: "
        f"{len(index):,} unique trigrams."
    )


build_inverted_index()

Indexed 600 documents: 99,862 unique trigrams.


In [8]:
with sqlite3.connect(DATABASE_PATH) as connection:
    top_20 = connection.execute(
        "SELECT ngram, filenames, Total "
        "FROM inverted_index "
        "ORDER BY Total DESC, ngram ASC "
        "LIMIT 20"
    ).fetchall()

print(f"{'Rank':>4} {'Total':>8}  Trigram | Filename occurrences")
for rank, (ngram, filenames, total) in enumerate(top_20, start=1):
    print(f"{rank:>4} {total:>8}  {ngram} | {filenames}")

Rank    Total  Trigram | Filename occurrences
   1       43  جمهوری اسلامی ایران | HAM2-750409-065.txt(4)+HAM2-750402-071.txt(3)+HAM2-750402-062.txt(2)+HAM2-750403-060.txt(2)+HAM2-750403-062.txt(2)+HAM2-750405-071.txt(2)+HAM2-750402-012.txt(1)+HAM2-750402-022.txt(1)+HAM2-750402-025.txt(1)+HAM2-750402-030.txt(1)+HAM2-750402-060.txt(1)+HAM2-750402-061.txt(1)+HAM2-750403-010.txt(1)+HAM2-750403-068.txt(1)+HAM2-750403-076.txt(1)+HAM2-750404-006.txt(1)+HAM2-750404-053.txt(1)+HAM2-750404-067.txt(1)+HAM2-750405-011.txt(1)+HAM2-750405-068.txt(1)+HAM2-750405-069.txt(1)+HAM2-750406-015.txt(1)+HAM2-750406-042.txt(1)+HAM2-750406-061.txt(1)+HAM2-750406-069.txt(1)+HAM2-750407-019.txt(1)+HAM2-750407-038.txt(1)+HAM2-750407-047.txt(1)+HAM2-750409-018.txt(1)+HAM2-750409-047.txt(1)+HAM2-750409-052.txt(1)+HAM2-750409-073.txt(1)+HAM2-750409-078.txt(1)+HAM2-750409-079.txt(1)
   2       41  خبرگزاری جمهوری اسلامی | HAM2-750402-030.txt(2)+HAM2-750404-010.txt(2)+HAM2-750402-013.txt(1)+HAM2-750402-033.txt(1)+HAM